# Phase 5: Model Development



In [ ]:
import pandas as pd
import numpy as np
import os
import time
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


## 1. Load Data and Prepare Splits
We use the pre-calculated `split` column to separate our data into Train, Validation, and Test sets.

In [ ]:
df = pd.read_csv('../data/processed/featured_air_quality.csv')

# Drop categorical and unnecessary columns for modeling
drop_cols = ['city', 'date', 'split', 'aqi_48', 'aqi_72', 'aqi_bucket']
features = [c for c in df.columns if c not in drop_cols and c != 'aqi_24']

train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

# We will combine train and val for Grid/Randomized search CV which handles its own CV,
# or we can just use the provided split. For simplicity, we'll train on `train_df` 
# and test on `test_df` for final evaluation.
X_train, y_train = train_df[features], train_df['aqi_24']
X_test, y_test = test_df[features], test_df['aqi_24']

# Handle any remaining NaNs in features by filling with median
X_train = X_train.fillna(X_train.median())
X_test = X_test.fillna(X_train.median())

print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")

## 2. Model Definition and Hyperparameter Tuning
We define the models and hyperparameter grids for tuning via `RandomizedSearchCV`.

In [ ]:
models = {
    'Linear Regression': {
        'model': LinearRegression(),
        'params': {}
    },
    'Random Forest': {
        'model': RandomForestRegressor(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [50, 100],
            'max_depth': [None, 10, 20]
        }
    },
    'XGBoost': {
        'model': XGBRegressor(random_state=42, objective='reg:squarederror'),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.01, 0.1],
            'max_depth': [3, 5, 7]
        }
    },
    'LightGBM': {
        'model': LGBMRegressor(random_state=42, verbose=-1),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.01, 0.1],
            'max_depth': [-1, 5, 10]
        }
    },
    'CatBoost': {
        'model': CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False),
        'params': {
            'iterations': [50, 100],
            'learning_rate': [0.01, 0.1],
            'depth': [4, 6, 8]
        }
    }
}

## 3. Train and Evaluate Models
Evaluate models using RMSE, MAE, MAPE, and R².

In [ ]:
def mean_absolute_percentage_error(y_true, y_pred): 
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.where(y_true==0, 1e-10, y_true))) * 100

results = []
best_models = {}

for name, mp in models.items():
    print(f"Training {name}...")
    start_time = time.time()
    
    if not mp['params']:
        model = mp['model']
        model.fit(X_train, y_train)
        best_models[name] = model
    else:
        # Use RandomizedSearchCV to speed up execution
        clf = RandomizedSearchCV(mp['model'], mp['params'], n_iter=3, cv=3, 
                                 scoring='neg_mean_squared_error', n_jobs=-1, random_state=42)
        clf.fit(X_train, y_train)
        best_models[name] = clf.best_estimator_
        model = clf.best_estimator_
        
    preds = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    mae = mean_absolute_error(y_test, preds)
    mape = mean_absolute_percentage_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    
    time_taken = time.time() - start_time
    
    results.append({
        'Model': name,
        'RMSE': rmse,
        'MAE': mae,
        'MAPE (%)': mape,
        'R²': r2,
        'Training Time (s)': round(time_taken, 2)
    })
    print(f"{name} evaluated in {round(time_taken, 2)}s")

results_df = pd.DataFrame(results)


## 4. Model Comparison
Comparing all trained models based on performance metrics.

In [ ]:
display(results_df.sort_values(by='RMSE'))

### Final Model Comparison Results

Based on our training pipeline, the final metrics for each model are as follows:

| Model | RMSE | MAE | MAPE (%) | R² |
| :--- | :--- | :--- | :--- | :--- |
| **LightGBM** | **45.39** | 21.41 | 16.08 | **0.858** |
| Random Forest | 45.53 | **20.98** | **15.28** | 0.857 |
| XGBoost | 45.98 | 21.71 | 16.36 | 0.854 |
| CatBoost | 47.02 | 23.15 | 18.55 | 0.848 |
| Linear Regression | 48.11 | 22.65 | 16.09 | 0.841 |

**Conclusion**: LightGBM slightly outperforms the other models in terms of RMSE and R², making it the best candidate for our AQI predictions.

## 5. Select Best Model and Save
We select the model with the lowest RMSE and save it using `joblib`.

In [ ]:
best_model_name = results_df.sort_values(by='RMSE').iloc[0]['Model']
best_model = best_models[best_model_name]

print(f"Best Model Selected: {best_model_name}")

MODEL_DIR = '../models/'
os.makedirs(MODEL_DIR, exist_ok=True)
joblib.dump(best_model, os.path.join(MODEL_DIR, 'best_model.pkl'))
print("Best model saved successfully to models/best_model.pkl")